In [14]:
import pandas as pd 

df = pd.read_csv("../data/raw/Loan_default.csv")
df.head()

,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


In [15]:
df.shape

(255347, 18)

The dataset contains 255,347 loan records and 18 features.  
This is a moderately large tabular dataset, which is sufficient for training machine learning models and performing reliable cross-validation.

In [16]:
df.columns

Index(['LoanID', 'Age', 'Income', 'LoanAmount', 'CreditScore',
       'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm',
       'DTIRatio', 'Education', 'EmploymentType', 'MaritalStatus',
       'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner',
       'Default'],
      dtype='object')

The dataset contains demographic, financial, and loan-related attributes.  
The target variable is `Default`, which indicates whether a borrower defaulted on the loan.

The column `LoanID` appears to be a unique identifier and does not carry predictive information, so it will be removed during preprocessing.

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 255347 entries, 0 to 255346
Data columns (total 18 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   LoanID          255347 non-null  object 
 1   Age             255347 non-null  int64  
 2   Income          255347 non-null  int64  
 3   LoanAmount      255347 non-null  int64  
 4   CreditScore     255347 non-null  int64  
 5   MonthsEmployed  255347 non-null  int64  
 6   NumCreditLines  255347 non-null  int64  
 7   InterestRate    255347 non-null  float64
 8   LoanTerm        255347 non-null  int64  
 9   DTIRatio        255347 non-null  float64
 10  Education       255347 non-null  object 
 11  EmploymentType  255347 non-null  object 
 12  MaritalStatus   255347 non-null  object 
 13  HasMortgage     255347 non-null  object 
 14  HasDependents   255347 non-null  object 
 15  LoanPurpose     255347 non-null  object 
 16  HasCoSigner     255347 non-null  object 
 17  Default   

The dataset contains three types of features:

• Numerical features: Age, Income, LoanAmount, CreditScore, MonthsEmployed, NumCreditLines, InterestRate, LoanTerm, DTIRatio  
• Categorical features: Education, EmploymentType, MaritalStatus, HasMortgage, HasDependents, LoanPurpose, HasCoSigner  
• Identifier: LoanID

The dataset has no missing values across all columns, which simplifies preprocessing since imputation is not required.

Categorical variables will need encoding (e.g., OneHotEncoder) before training machine learning models.

In [18]:
df.describe()

,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Default
count,255347.000000,255347.000000,255347.000000,255347.000000,255347.000000,255347.000000,255347.000000,255347.000000,255347.000000,255347.000000
mean,43.498306,82499.304597,127578.865512,574.264346,59.541976,2.501036,13.492773,36.025894,0.500212,0.116128
std,14.990258,38963.013729,70840.706142,158.903867,34.643376,1.117018,6.636443,16.969330,0.230917,0.320379
min,18.000000,15000.000000,5000.000000,300.000000,0.000000,1.000000,2.000000,12.000000,0.100000,0.000000
25%,31.000000,48825.500000,66156.000000,437.000000,30.000000,2.000000,7.770000,24.000000,0.300000,0.000000
50%,43.000000,82466.000000,127556.000000,574.000000,60.000000,2.000000,13.460000,36.000000,0.500000,0.000000
75%,56.000000,116219.000000,188985.000000,712.000000,90.000000,3.000000,19.250000,48.000000,0.700000,0.000000
max,69.000000,149999.000000,249999.000000,849.000000,119.000000,4.000000,25.000000,60.000000,0.900000,1.000000


The summary statistics reveal the distribution of the numerical features.

Key observations:

• The average borrower age is around 43 years, with values ranging from 18 to 69.  
• The average annual income is approximately $82,499.  
• Loan amounts vary widely, from $5,000 to $249,999.  
• Credit scores range from 300 to 849, covering the typical credit scoring range.

These variations suggest that feature scaling will be useful for algorithms that rely on distance calculations (e.g., KNN or SVM).

In [19]:
df["Default"].value_counts()

Default
0    225694
1     29653
Name: count, dtype: int64

The dataset is imbalanced.

• Non-default cases: 225,694 (~88%)
• Default cases: 29,653 (~12%)

Since the classes are imbalanced, accuracy alone may not be an appropriate evaluation metric.  
Metrics such as precision, recall, F1-score, and ROC-AUC will provide better insight into model performance.

In the context of loan risk prediction, recall for the default class is particularly important because failing to detect a risky borrower could result in financial loss.

In [20]:
df["Default"].value_counts(normalize=True)

Default
0    0.883872
1    0.116128
Name: proportion, dtype: float64

The target variable distribution shows that the dataset is imbalanced.

Approximately 88% of loans are non-defaults while about 12% are defaults.

Because of this imbalance, accuracy alone may not be sufficient to evaluate model performance. Metrics such as precision, recall, F1-score, and ROC-AUC will also be used to assess the models.

## Removing Identifier Column

In [21]:
df = df.drop(columns="LoanID")

## Separate Features and Target

In [22]:
X = df.drop(columns="Default")
y = df["Default"]

X.shape, y.shape

((255347, 16), (255347,))

In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,y,
    test_size=0.2,
    random_state=42, 
    stratify=y
)

X_train.shape, X_test.shape

((204277, 16), (51070, 16))

The dataset is split into training and testing sets.

• 80% of the data is used for training the models
• 20% is reserved for evaluating model performance on unseen data

Stratified splitting ensures that the proportion of default and non-default cases remains consistent in both the training and test sets.

In [24]:
y_train.value_counts(normalize=True) , y_test.value_counts(normalize=True)

(Default
 0    0.883873
 1    0.116127
 Name: proportion, dtype: float64,
 Default
 0    0.883865
 1    0.116135
 Name: proportion, dtype: float64)

The class distribution remains consistent between the training and testing sets due to stratified sampling.


## Identify Numerical and Categorical Features

In [25]:
num_features = X.select_dtypes(include=["int64", "float64"]).columns
cat_features = X.select_dtypes(include=["object"]).columns
num_features, cat_features

(Index(['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed',
        'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio'],
       dtype='object'),
 Index(['Education', 'EmploymentType', 'MaritalStatus', 'HasMortgage',
        'HasDependents', 'LoanPurpose', 'HasCoSigner'],
       dtype='object'))

# Build Preprocessing Pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("num",StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"),cat_features)
    ]
)


Different preprocessing steps are applied to numerical and categorical features using ColumnTransformer.

• Numerical features are scaled using StandardScaler.
• Categorical features are encoded using OneHotEncoder.

This ensures that all features are converted into a suitable numerical format before being passed to the machine learning model.

# Build the Pipeline

In [29]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

# Define Hyperparameter Grid

In [30]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "model__C": [0.01, 0.1, 1, 10]
}

Hyperparameter tuning is performed to identify the best regularization strength (C) for Logistic Regression.

The parameter C controls the tradeoff between fitting the training data well and keeping the model simple to avoid overfitting.

# Grid Search

In [31]:
grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step..._iter=1000))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [0.01, 0.1, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed;- >3 : the fold and candidate para

GridSearchCV performs cross-validation to evaluate multiple hyperparameter combinations.

Five-fold cross-validation is used, meaning the training data is split into five subsets and the model is trained and validated multiple times.

The F1-score is used as the evaluation metric because the dataset is imbalanced.

# Best Model

In [32]:
print("Best Parameters", grid.best_params_)
print("Best CV score:", grid.best_score_)

Best Parameters {'model__C': 1}
Best CV score: 0.06420837478028589


The best hyperparameters are selected based on the highest cross-validation F1-score.

This model will be used for further evaluation on the unseen test dataset.

In [33]:
best_model = grid.best_estimator_

y_train_pred = best_model.predict(X_train)
y_test_pred = best_model.predict(X_test)

# Evaluate Model

In [34]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

precision = precision_score(y_test, y_test_pred)
recall = recall_score(y_test, y_test_pred)
f1 = f1_score(y_test, y_test_pred)

print("Train accuracy", train_acc)
print("Test accuracy", test_acc)

print("Precision", precision)
print("Recall", recall)
print("F1-score", f1)

Train accuracy 0.885102091767551
Test accuracy 0.885275112590562
Precision 0.608433734939759
Recall 0.03405833754847412
F1-score 0.06450582787801373


The training and testing accuracies are nearly identical (~88.5%), indicating that the model is not overfitting.

However, the high accuracy is misleading due to class imbalance. Since approximately 88% of loans are non-defaults, a model that predicts most cases as non-default can still achieve high accuracy without effectively identifying risky borrowers.

# Confusion Matrix

In [35]:
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_test, y_test_pred))
print(classification_report(y_test, y_test_pred))

[[45009   130]
 [ 5729   202]]
              precision    recall  f1-score   support

           0       0.89      1.00      0.94     45139
           1       0.61      0.03      0.06      5931

    accuracy                           0.89     51070
   macro avg       0.75      0.52      0.50     51070
weighted avg       0.85      0.89      0.84     51070

